In [1]:
import numpy as np
import pandas as pd
import openpyxl
import re
import umap
import matplotlib.pyplot as plt
import seaborn as sns
import datetime
from collections import Counter
pd.set_option('display.max_columns', 200)

# Metadata QC

In [2]:
# Looking at demographics
metadata = pd.DataFrame()

In [3]:
file = "data_from_collaborators/Saliva database_clinical_Max_US.xlsx"
# ['List_of_symptoms', 'Cases symptoms', 'Controls symptoms', 'Cases demographics', 'Control demographics']
# file = "data_from_collaborators/Catalina Copy of Original Demographics-Cardiff Long COVID.xlsx"
file = openpyxl.load_workbook(file, data_only = True)
print(file.sheetnames)
sheet = "Cases demographics"
# sheet = "Control demographics"
data = pd.DataFrame(file[sheet].values)
data = data.fillna(float("nan"))
data = data.dropna(axis = 0, how = "all")
data = data.dropna(axis = 1, how = "all")
data = data.astype(str)
columns = []
for col in data.iloc[0,:]: 
    col = col.replace(" ", "_").replace(".", "_").lower()
    while "__" in col: col = col.replace("__", "_")
    if col[-1] == "_": col = col[:-1]
    columns.append(col)
data.columns = columns
data = data.rename(columns = {"_id": "id", "gender": "sex"})
data = data.iloc[1:, :].reset_index(drop = True)
# Data QC
data["ethnicity"] = [val.lower() for val in data["ethnicity"]]
print(list(np.unique(data["ethnicity"])))
values = []
for val in data["smoking"]: 
    if "never" in val.lower(): values.append("0")
    else: values.append("1")
data["smoking"] = values
# data["smoking"] = [val.lower() for val in data["smoking"]]
print(list(np.unique(data["smoking"])))
values = []
for val in data["covid_vaccination"]: 
    if "yes" in val.lower(): values.append("1")
    else: values.append("0")
data["covid_vaccination"] = values
values = []
for val in data["sex"]: 
    if "woman" in val.lower() or "f" in val.lower(): values.append("F")
    elif "man" in val.lower() or "m" in val.lower(): values.append("M")
    else: values.append("nan")
data["sex"] = values
del data["employment"]
del data["education"]
# Fixing id
data["id"] = [val.replace("0.", "CO").replace("1.", "CA") for val in data["id"]]
values = []
for val in data["id"]: 
    if len(val) == 4: values.append(val + "0")
    else: values.append(val)
data["id"] = values
print(list(np.unique(data["id"])))
metadata = pd.concat([metadata, data])
data

['List_of_symptoms', 'Cases symptoms', 'Controls symptoms', 'Cases demographics', 'Control demographics']
['white']
['0', '1']
['CA001', 'CA003', 'CA004', 'CA009', 'CA012', 'CA013', 'CA015', 'CA016', 'CA019', 'CA025', 'CA026', 'CA027', 'CA029', 'CA034', 'CA035', 'CA037', 'CA039', 'CA042', 'CA045', 'CA048', 'CA052', 'CA055', 'CA056', 'CA060', 'CA064', 'CA065', 'CA067', 'CA069', 'CA070', 'CA073', 'CA074', 'CA076', 'CA078', 'CA081', 'CA095', 'CA101', 'CA121', 'CA130', 'CA132']


,id,ethnicity,smoking,body_mass_index,covid_vaccination,sex,age
0,CA001,white,0,25.7,1,M,44
1,CA012,white,0,29.4,1,F,45
2,CA013,white,1,45,1,F,42
3,CA009,white,0,24,1,M,33
4,CA027,white,0,29.7,1,F,48
5,CA019,white,0,33.3,1,M,49
6,CA029,white,0,30.5,1,F,40
7,CA003,white,0,30.9,0,F,35
8,CA004,white,0,22.9,1,F,45
9,CA034,white,0,26.7,1,M,38


In [4]:
file = "data_from_collaborators/Saliva database_clinical_Max_US.xlsx"
# ['List_of_symptoms', 'Cases symptoms', 'Controls symptoms', 'Cases demographics', 'Control demographics']
# file = "Catalina Copy of Original Demographics-Cardiff Long COVID.xlsx"
file = openpyxl.load_workbook(file, data_only = True)
print(file.sheetnames)
sheet = "Control demographics"
data = pd.DataFrame(file[sheet].values)
data = data.fillna(float("nan"))
data = data.dropna(axis = 0, how = "all")
data = data.dropna(axis = 1, how = "all")
data = data.astype(str)
columns = []
for col in data.iloc[0,:]: 
    col = col.replace(" ", "_").replace(".", "_").lower()
    while "__" in col: col = col.replace("__", "_")
    if col[-1] == "_": col = col[:-1]
    columns.append(col)
data.columns = columns
data = data.rename(columns = {"_id": "id", "gender": "sex"})
data = data.iloc[1:, :].reset_index(drop = True)
# Data QC
data["ethnicity"] = [val.lower() for val in data["ethnicity"]]
print(list(np.unique(data["ethnicity"])))
values = []
for val in data["smoking"]: 
    if "never" in val.lower(): values.append("0")
    else: values.append("1")
data["smoking"] = values
# data["smoking"] = [val.lower() for val in data["smoking"]]
print(list(np.unique(data["smoking"])))
values = []
for val in data["covid_vaccination"]: 
    if "yes" in val.lower(): values.append("1")
    else: values.append("0")
data["covid_vaccination"] = values
values = []
for val in data["sex"]: 
    if "woman" in val.lower() or "f" in val.lower(): values.append("F")
    elif "man" in val.lower() or "m" in val.lower(): values.append("M")
    else: values.append("nan")
data["sex"] = values
del data["employment"]
del data["education"]
# Fixing id
data["id"] = [val.replace("0.", "CO").replace("1.", "CA") for val in data["id"]]
values = []
for val in data["id"]: 
    if len(val) == 4: values.append(val + "0")
    else: values.append(val)
data["id"] = values
print(list(np.unique(data["id"])))
metadata = pd.concat([metadata, data])
data

['List_of_symptoms', 'Cases symptoms', 'Controls symptoms', 'Cases demographics', 'Control demographics']
['asian', 'mixed-white asian ', 'white']
['0', '1']
['CO002', 'CO004', 'CO005', 'CO007', 'CO008', 'CO011', 'CO013', 'CO014', 'CO016', 'CO022', 'CO029', 'CO032', 'CO036', 'CO037', 'CO055', 'CO057', 'CO058', 'CO060', 'CO063', 'CO064', 'CO066', 'CO069', 'CO071', 'CO073', 'CO074', 'CO077', 'CO078', 'CO080', 'CO082', 'CO083', 'CO084', 'CO085', 'CO087', 'CO091', 'CO092', 'CO093', 'CO099', 'CO104', 'CO106', 'CO114', 'CO116', 'CO119', 'CO121']


,id,ethnicity,smoking,body_mass_index,covid_vaccination,sex,age
0,CO005,white,0,27.6,1,F,38
1,CO099,white,0,27.8,1,F,40
2,CO063,white,0,34.7,1,M,29
3,CO093,white,0,26.9,1,M,29
4,CO106,white,0,25.5,1,F,54
5,CO119,white,0,25.9,1,M,29
6,CO066,white,0,31.2,1,F,51
7,CO116,white,0,30,1,F,39
8,CO036,white,1,28,1,F,61
9,CO078,white,0,36,1,F,32


In [5]:
# ['List_of_symptoms', 'Cases symptoms', 'Controls symptoms', 'Cases demographics', 'Control demographics']
file = "data_from_collaborators/Catalina Copy of Original Demographics-Cardiff Long COVID.xlsx"
# ['Cases', 'Controls']
file = openpyxl.load_workbook(file, data_only = True)
print(file.sheetnames)
sheet = "Cases"
# sheet = "Controls"
data = pd.DataFrame(file[sheet].values)
data = data.fillna(float("nan"))
data = data.dropna(axis = 0, how = "all")
data = data.dropna(axis = 1, how = "all")
data = data.astype(str)
columns = []
for col in data.iloc[0,:]: 
    col = col.replace(" ", "_").replace(".", "_").lower()
    while "__" in col: col = col.replace("__", "_")
    if col[-1] == "_": col = col[:-1]
    columns.append(col)
data.columns = columns
data = data.rename(columns = {"study_id": "id", "_id": "id"})
data = data.iloc[1:, :].reset_index(drop = True)
data = data[data["id"] != "nan"].reset_index(drop = True)
data = data[["id", "sex", "age", "ethnicity", "height", "weight"]]
# Data QC
data["ethnicity"] = [val.lower() for val in data["ethnicity"]]
print(list(np.unique(data["ethnicity"])))
values = []
for val in data["sex"]: 
    if "woman" in val.lower() or "f" in val.lower(): values.append("F")
    elif "man" in val.lower() or "m" in val.lower(): values.append("M")
    else: values.append("nan")
data["sex"] = values
print(list(np.unique(data["id"])))
metadata = pd.concat([metadata, data])
data

['Cases', 'Controls']
['arab', 'arab welsh', 'asian', 'asian filipino', 'asian, indian', 'asian/chinese', 'black', 'black african british', 'black/white british', 'british white', 'caucasian', 'european', 'filipino', 'middle eastern', 'mixed', 'muslim', 'welsh british', 'white (other)', 'white british']
['CA001', 'CA002', 'CA003', 'CA004', 'CA005', 'CA006', 'CA007', 'CA008', 'CA009', 'CA010', 'CA011', 'CA012', 'CA013', 'CA014', 'CA015', 'CA016', 'CA017', 'CA018', 'CA019', 'CA020', 'CA021', 'CA022', 'CA023', 'CA024', 'CA025', 'CA026', 'CA027', 'CA028', 'CA029', 'CA030', 'CA031', 'CA032', 'CA033', 'CA034', 'CA035', 'CA036', 'CA037', 'CA038', 'CA039', 'CA040', 'CA041', 'CA042', 'CA043', 'CA044', 'CA045', 'CA046', 'CA047', 'CA048', 'CA049', 'CA050', 'CA051', 'CA052', 'CA053', 'CA054', 'CA055', 'CA056', 'CA057', 'CA058', 'CA059', 'CA060', 'CA061', 'CA062', 'CA063', 'CA064', 'CA065', 'CA066', 'CA067', 'CA068', 'CA069', 'CA070', 'CA071', 'CA072', 'CA073', 'CA074', 'CA075', 'CA076', 'CA077', '

,id,sex,age,ethnicity,height,weight
0,CA001,M,42.0,caucasian,167.0,79.6
1,CA002,F,46.0,caucasian,153.5,124.6
2,CA003,F,35.0,caucasian,165.0,68.0
3,CA004,F,45.0,caucasian,163.0,55.0
4,CA005,F,52.0,caucasian,nan,nan
...,...,...,...,...,...,...
212,CA213,F,48.0,caucasian,165.0,70.0
213,CA214,F,59.0,caucasian,168.0,73.0
214,CA215,F,60.0,caucasian,168.0,115.0
215,CA216,F,40.0,caucasian,165.0,50.0


In [6]:
# ['List_of_symptoms', 'Cases symptoms', 'Controls symptoms', 'Cases demographics', 'Control demographics']
file = "data_from_collaborators/Catalina Copy of Original Demographics-Cardiff Long COVID.xlsx"
# ['Cases', 'Controls']
file = openpyxl.load_workbook(file, data_only = True)
print(file.sheetnames)
sheet = "Cases"
sheet = "Controls"
data = pd.DataFrame(file[sheet].values)
data = data.fillna(float("nan"))
data = data.dropna(axis = 0, how = "all")
data = data.dropna(axis = 1, how = "all")
data = data.astype(str)
columns = []
for col in data.iloc[0,:]: 
    col = col.replace(" ", "_").replace(".", "_").lower()
    while "__" in col: col = col.replace("__", "_")
    if col[-1] == "_": col = col[:-1]
    columns.append(col)
data.columns = columns
data = data.rename(columns = {"study_id": "id", "_id": "id"})
data = data.iloc[1:, :].reset_index(drop = True)
data = data[data["id"] != "nan"].reset_index(drop = True)
data = data[["id", "sex", "age", "ethnicity", "height", "weight"]]
# Data QC
data["ethnicity"] = [val.lower() for val in data["ethnicity"]]
print(list(np.unique(data["ethnicity"])))
values = []
for val in data["sex"]: 
    if "woman" in val.lower() or "f" in val.lower(): values.append("F")
    elif "man" in val.lower() or "m" in val.lower(): values.append("M")
    else: values.append("nan")
data["sex"] = values
print(list(np.unique(data["id"])))
data["id"] = [val.replace("C0", "CO").replace("CO093", "C0093") for val in data["id"]]
metadata = pd.concat([metadata, data])
data

['Cases', 'Controls']
['asian', 'asian chinese', 'caucasian', 'egyptian', 'filipino', 'indian', 'middle eastern', 'mixed white/asian', 'mixed white/black']
['C0078', 'C0079', 'CO001', 'CO002', 'CO003', 'CO004', 'CO005', 'CO006', 'CO007', 'CO008', 'CO009', 'CO010', 'CO011', 'CO012', 'CO013', 'CO014', 'CO015', 'CO016', 'CO017', 'CO018', 'CO019', 'CO020', 'CO021', 'CO022', 'CO023', 'CO024', 'CO025', 'CO026', 'CO027', 'CO028', 'CO029', 'CO030', 'CO031', 'CO032', 'CO033', 'CO034', 'CO035', 'CO036', 'CO037', 'CO038', 'CO039', 'CO040', 'CO041', 'CO042', 'CO043', 'CO044', 'CO045', 'CO046', 'CO047', 'CO048', 'CO049', 'CO050', 'CO051', 'CO052', 'CO053', 'CO054', 'CO055', 'CO056', 'CO057', 'CO058', 'CO059', 'CO060', 'CO061', 'CO062', 'CO063', 'CO064', 'CO065', 'CO066', 'CO067', 'CO068', 'CO069', 'CO070', 'CO071', 'CO072', 'CO073', 'CO074', 'CO075', 'CO076', 'CO077', 'CO080', 'CO081', 'CO082', 'CO083', 'CO084', 'CO085', 'CO086', 'CO087', 'CO088', 'CO089', 'CO090', 'CO091', 'CO092', 'CO093', 'CO094

,id,sex,age,ethnicity,height,weight
0,CO001,F,54.0,caucasian,162.5,64.8
1,CO002,M,58.0,caucasian,172.5,101.4
2,CO003,F,23.0,caucasian,157.5,65.3
3,CO004,F,58.0,caucasian,165.0,85.0
4,CO005,F,37.0,caucasian,170.0,79.0
...,...,...,...,...,...,...
113,CO114,F,58.0,caucasian,163.0,82.0
114,CO115(a),M,61.0,caucasian,183.0,92.0
115,CO116,F,39.0,caucasian,161.0,62.0
116,CO117,M,24.0,caucasian,190.0,110.0


In [7]:
metadata["id"] = [val.replace("CO115(a)", "CO115") for val in metadata["id"]]
print(metadata[metadata["sex"] == "nan"].shape[0], "nans")
metadata = metadata[metadata["sex"] != "nan"]
metadata = metadata.sort_values(["id", "sex"]).reset_index(drop = True)
metadata = metadata.drop_duplicates(["id"], keep = "first").sort_values(["id", "sex"]).reset_index(drop = True)
print(list(np.unique(metadata["id"])))
# metadata.to_csv("data_cleaned/metadata_cardiff_n338.csv", index = False)
metadata

0 nans
['C0093', 'CA001', 'CA002', 'CA003', 'CA004', 'CA005', 'CA006', 'CA007', 'CA008', 'CA009', 'CA010', 'CA011', 'CA012', 'CA013', 'CA014', 'CA015', 'CA016', 'CA017', 'CA018', 'CA019', 'CA020', 'CA021', 'CA022', 'CA023', 'CA024', 'CA025', 'CA026', 'CA027', 'CA028', 'CA029', 'CA030', 'CA031', 'CA032', 'CA033', 'CA034', 'CA035', 'CA036', 'CA037', 'CA038', 'CA039', 'CA040', 'CA041', 'CA042', 'CA043', 'CA044', 'CA045', 'CA046', 'CA047', 'CA048', 'CA049', 'CA050', 'CA051', 'CA052', 'CA053', 'CA054', 'CA055', 'CA056', 'CA057', 'CA058', 'CA059', 'CA060', 'CA061', 'CA062', 'CA063', 'CA064', 'CA065', 'CA066', 'CA067', 'CA068', 'CA069', 'CA070', 'CA071', 'CA072', 'CA073', 'CA074', 'CA075', 'CA076', 'CA077', 'CA078', 'CA079', 'CA080', 'CA081', 'CA082', 'CA083', 'CA084', 'CA085', 'CA086', 'CA087', 'CA088', 'CA089', 'CA090', 'CA091', 'CA092', 'CA093', 'CA094', 'CA095', 'CA096', 'CA097', 'CA098', 'CA099', 'CA100', 'CA101', 'CA102', 'CA103', 'CA104', 'CA105', 'CA106', 'CA107', 'CA108', 'CA109', 'C

,id,ethnicity,smoking,body_mass_index,covid_vaccination,sex,age,height,weight
0,C0093,caucasian,NaN,NaN,NaN,M,28.0,168.0,78.6
1,CA001,white,0,25.7,1,M,44,NaN,NaN
2,CA002,caucasian,NaN,NaN,NaN,F,46.0,153.5,124.6
3,CA003,white,0,30.9,0,F,35,NaN,NaN
4,CA004,white,0,22.9,1,F,45,NaN,NaN
...,...,...,...,...,...,...,...,...,...
333,CO116,white,0,30,1,F,39,NaN,NaN
334,CO117,caucasian,NaN,NaN,NaN,M,24.0,190.0,110.0
335,CO118,caucasian,NaN,NaN,NaN,M,35.0,178.0,80.0
336,CO119,white,0,25.9,1,M,29,NaN,NaN


In [8]:
metadata["id"].value_counts()

C0093    1
CO005    1
CO013    1
CO012    1
CO011    1
        ..
CA111    1
CA110    1
CA109    1
CA108    1
CO121    1
Name: id, Length: 338, dtype: int64

In [9]:
metadata["sex"].value_counts()

F    249
M     89
Name: sex, dtype: int64

# Loading data

In [10]:
df = pd.DataFrame()
for file in ["data_from_collaborators/Cardiff_case_control_symptoms_final_JCVI.xlsx", 
             "data_from_collaborators/Cardiff_Missing_Data_JCVI_2nd.xlsx"]: 
    print(file)
    file = openpyxl.load_workbook(file, data_only = True)
    print(file.sheetnames)
    sheets = ['Cases', 'Control ', 'Control']
    for sheet in sheets:
        if sheet not in list(file.sheetnames): 
            continue
        data = pd.DataFrame(file[sheet].values)
        data = data.fillna(float("nan"))
        columns = []
        for col in data.iloc[0,:]: 
            col = col.replace(" ", "_").replace(".", "_").lower()
            while "__" in col: col = col.replace("__", "_")
            if col[-1] == "_": col = col[:-1]
            columns.append(col)
        data.columns = columns
        data = data.iloc[1:, :].reset_index(drop = True)
        if "group" not in list(data.columns): 
            values = []
            for val in data["id"]: 
                if "CA" in val: values.append("Case")
                elif "CO" in val: values.append("Control")
            data["group"] = values
#         print(np.unique(data["id"]))
        data["id"] = data["id"].astype(str)
        # standardizing strings
        def transform_value(value): return str(value).lower().replace(" ", "_").replace(",", "").replace("(", "").replace(")", "")
        # data = data.applymap(transform_value).reset_index(drop = True)
        df = pd.concat([df, data])
data = df.copy()
data["id"] = [val.replace("0.093", "COO93").replace("0.", "CO").replace("1.", "CA") for val in data["id"]]
# data = data.drop_duplicates(keep = "first").sort_values("id").reset_index(drop = True)
data

data_from_collaborators/Cardiff_case_control_symptoms_final_JCVI.xlsx
['cardiff_case_control_symptoms', 'Cases', 'Control ']
data_from_collaborators/Cardiff_Missing_Data_JCVI_2nd.xlsx
['Cases', 'Control']


,id,group,chest_pain,palpitations_heart_racing,poor_concentration,breathlessness_shortness_of_breath,cough,fevers_did_you_feel_hot,headache,abdominal_pain,change_in_smell,dizziness_or_vertigo,muscle_pain,poor_balance_unsteadiness,sleep_disturbance,increased_anxiety_worry,low_mood_lack_of_enjoyment_in_normal_activities,nausea,vomiting,muscle_cramps,constipation,diarrhoea,feeling_faint_or_light_headed,loss_of_appetite,back_pain,chills_do_you_feel_unusually_cold,numbness_odd_or_loss_of_sensation,increased_body_odour_sweating,general_body_pain_e_g_arms_legs_or_joints,pain_during_sex_having_intercourse,rash,runny_or_congested_nose,sore_throat,high_temperature_when_you_need_it_measured,change_in_vision,fatigue
0,CA001,Case,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0
1,CA002,Case,1,1,0,1,1,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
2,CA003,Case,1,1,1,1,1,0,1,0,1,1,0,0,1,0,0,1,1,0,0,0,1,1,0,0,0,1,0,0,1,1,0,0,0,1
3,CA004,Case,0,0,0,0,0,0,1,1,0,1,0,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1
4,CA005,Case,1,1,1,1,1,0,0,0,0,1,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22,CO112,Control,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
23,CO113,Control,1,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
24,CO115,Control,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
25,CO117,Control,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [11]:
data[data.duplicated("id", keep = False)].sort_values("id")

,id,group,chest_pain,palpitations_heart_racing,poor_concentration,breathlessness_shortness_of_breath,cough,fevers_did_you_feel_hot,headache,abdominal_pain,change_in_smell,dizziness_or_vertigo,muscle_pain,poor_balance_unsteadiness,sleep_disturbance,increased_anxiety_worry,low_mood_lack_of_enjoyment_in_normal_activities,nausea,vomiting,muscle_cramps,constipation,diarrhoea,feeling_faint_or_light_headed,loss_of_appetite,back_pain,chills_do_you_feel_unusually_cold,numbness_odd_or_loss_of_sensation,increased_body_odour_sweating,general_body_pain_e_g_arms_legs_or_joints,pain_during_sex_having_intercourse,rash,runny_or_congested_nose,sore_throat,high_temperature_when_you_need_it_measured,change_in_vision,fatigue


In [12]:
# data.to_csv("data_cleaned/cardiff_data_all_n338.csv", index = False)

In [13]:
data["group"].value_counts()

Case       217
Control    121
Name: group, dtype: int64

In [14]:
data[data["id"].duplicated(keep = False)]

,id,group,chest_pain,palpitations_heart_racing,poor_concentration,breathlessness_shortness_of_breath,cough,fevers_did_you_feel_hot,headache,abdominal_pain,change_in_smell,dizziness_or_vertigo,muscle_pain,poor_balance_unsteadiness,sleep_disturbance,increased_anxiety_worry,low_mood_lack_of_enjoyment_in_normal_activities,nausea,vomiting,muscle_cramps,constipation,diarrhoea,feeling_faint_or_light_headed,loss_of_appetite,back_pain,chills_do_you_feel_unusually_cold,numbness_odd_or_loss_of_sensation,increased_body_odour_sweating,general_body_pain_e_g_arms_legs_or_joints,pain_during_sex_having_intercourse,rash,runny_or_congested_nose,sore_throat,high_temperature_when_you_need_it_measured,change_in_vision,fatigue


In [15]:
print(len(np.unique(data["id"])))
print(list(np.unique(data["id"])))

338
['CA001', 'CA002', 'CA003', 'CA004', 'CA005', 'CA006', 'CA007', 'CA008', 'CA009', 'CA01', 'CA011', 'CA012', 'CA013', 'CA014', 'CA015', 'CA016', 'CA017', 'CA018', 'CA019', 'CA02', 'CA021', 'CA022', 'CA023', 'CA024', 'CA025', 'CA026', 'CA027', 'CA028', 'CA029', 'CA03', 'CA031', 'CA032', 'CA033', 'CA034', 'CA035', 'CA036', 'CA037', 'CA038', 'CA039', 'CA04', 'CA041', 'CA042', 'CA043', 'CA044', 'CA045', 'CA046', 'CA047', 'CA048', 'CA049', 'CA05', 'CA051', 'CA052', 'CA053', 'CA054', 'CA055', 'CA056', 'CA057', 'CA058', 'CA059', 'CA06', 'CA061', 'CA062', 'CA063', 'CA064', 'CA065', 'CA066', 'CA067', 'CA068', 'CA069', 'CA07', 'CA071', 'CA072', 'CA073', 'CA074', 'CA075', 'CA076', 'CA077', 'CA078', 'CA079', 'CA08', 'CA081', 'CA082', 'CA083', 'CA084', 'CA085', 'CA086', 'CA087', 'CA088', 'CA089', 'CA09', 'CA091', 'CA092', 'CA093', 'CA094', 'CA095', 'CA096', 'CA097', 'CA098', 'CA099', 'CA1', 'CA101', 'CA102', 'CA103', 'CA104', 'CA105', 'CA106', 'CA107', 'CA108', 'CA109', 'CA11', 'CA111', 'CA112',

In [16]:
subset = data.copy()
subset[subset.duplicated(keep = False)].sort_values("id")

,id,group,chest_pain,palpitations_heart_racing,poor_concentration,breathlessness_shortness_of_breath,cough,fevers_did_you_feel_hot,headache,abdominal_pain,change_in_smell,dizziness_or_vertigo,muscle_pain,poor_balance_unsteadiness,sleep_disturbance,increased_anxiety_worry,low_mood_lack_of_enjoyment_in_normal_activities,nausea,vomiting,muscle_cramps,constipation,diarrhoea,feeling_faint_or_light_headed,loss_of_appetite,back_pain,chills_do_you_feel_unusually_cold,numbness_odd_or_loss_of_sensation,increased_body_odour_sweating,general_body_pain_e_g_arms_legs_or_joints,pain_during_sex_having_intercourse,rash,runny_or_congested_nose,sore_throat,high_temperature_when_you_need_it_measured,change_in_vision,fatigue


In [17]:
print(data[list(data.isna().sum(axis = 1) != 0)].shape)
data[list(data.isna().sum(axis = 1) != 0)]

(23, 36)


,id,group,chest_pain,palpitations_heart_racing,poor_concentration,breathlessness_shortness_of_breath,cough,fevers_did_you_feel_hot,headache,abdominal_pain,change_in_smell,dizziness_or_vertigo,muscle_pain,poor_balance_unsteadiness,sleep_disturbance,increased_anxiety_worry,low_mood_lack_of_enjoyment_in_normal_activities,nausea,vomiting,muscle_cramps,constipation,diarrhoea,feeling_faint_or_light_headed,loss_of_appetite,back_pain,chills_do_you_feel_unusually_cold,numbness_odd_or_loss_of_sensation,increased_body_odour_sweating,general_body_pain_e_g_arms_legs_or_joints,pain_during_sex_having_intercourse,rash,runny_or_congested_nose,sore_throat,high_temperature_when_you_need_it_measured,change_in_vision,fatigue
35,CA036,Case,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
111,CA112,Case,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
122,CA123,Case,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1
125,CA126,Case,1,1,1,1,1,0,0,0,0,1,1,1,1,1,1,0,0,0,0,0,1,0,0,0,0,0,NaN,0,0,0,0,0,0,1
2,CO003,Control,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,CO006,Control,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,CO008,Control,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,CO009,Control,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12,CO013,Control,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14,CO015,Control,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Averaging single missing points

In [18]:
# Changing the Cardiff naming to Sinai naming
file_ = "data_dictionary - Sheet1.csv"
data_dict = pd.read_csv(file_)
values = []
for col in data_dict["Cardiff Name"]: 
    col = str(col).replace(" ", "_").replace(".", "_").lower()
    while "__" in col: col = col.replace("__", "_")
    if col[-1] == "_": col = col[:-1]
    values.append(col)
data_dict["Cardiff Name"] = values
data_dict["Sinai Description"] = [str(val).lower() for val in data_dict["Sinai Description"]]
data_dict

,Organ System,Sinai Name,Sinai Description,Cardiff Name,Emory Name,UCSF Name,UCSF Description
0,Cognitive/Psych,currentsymptoms_27,"difficulty with concentration or reading or ""b...",poor_concentration,Brain Fog (3),concen,"trouble concentrating, trouble with your think..."
1,Cognitive/Psych,currentsymptoms_28,"confusion, difficulty thinking",nan,"Confusion, difficulty thinking",concen,"trouble concentrating, trouble with your think..."
2,Cognitive/Psych,currentsymptoms_30,memory problems or forgetfulness,nan,"Forgetful, memory problem",concen,"trouble concentrating, trouble with your think..."
3,Neurological,currentsymptoms_15,fatigue or tiredness,fatigue,Fatigue (1),fatig,feeling tired or having low energy
4,pulmonary,currentsymptoms_11,cough,cough,Cough,cough,cough
...,...,...,...,...,...,...,...
56,NaN,NaN,nan,nan,Thirst (3),NaN,NaN
57,NaN,NaN,nan,nan,Dry Eye,NaN,NaN
58,NaN,NaN,nan,nan,Rhinitis,NaN,NaN
59,Cognitive/Psych,NaN,nan,increased_anxiety_worry,Anxiety,NaN,NaN


In [19]:
file_ = open('symptoms_sinai.txt','r')
symptoms = file_.readlines()[:44]
file_.close()
symptoms = [val.lower().replace("\n", "") for val in symptoms]
print(len(symptoms), "symptoms")

file_ = open('Neurological_Symptom_Mapping.txt','r')
mapping = [val.lower().replace("\n", "").replace("\t", "") for val in file_.readlines()]
file_.close()
neurologicals = [val.lower() for val in ["Neurological", "Cognitive/Psych", "Musculo-Skeletal", "G.I", "Sexual/Hormonal Function", "sensory changes", "temperature regulation", "pulmonary", "cardiovascular"]]
neurological_symptom_mapping, color_mapping, symptoms_ordered, symptoms_ordered_colors = {}, {}, [], []
curr, ct = "", -1
for val in mapping: 
    val = val.replace("\t", "")
    if val in neurologicals: 
        curr = val
        ct += 1
    else: 
        if val not in symptoms: 
            if val == "abnormal changes in body temp": val = "Abnormal changes in body temperature".lower()
            elif val == "unexplained sweats or flushin": val = "Unexplained sweats or flushing".lower()
        symptoms_ordered.append(val)
        if curr not in list(neurological_symptom_mapping.keys()): neurological_symptom_mapping[curr] = [val]
        else: neurological_symptom_mapping[curr].append(val)
print(len(neurological_symptom_mapping.keys()),  "symptom groups")
neurological_symptom_mapping

44 symptoms
9 symptom groups


{'neurological': ['headache',
  'problems seeing (double or blurry vision)',
  'difficulty swallowing',
  'fatigue or tiredness',
  'dizziness or lightheadedness',
  'fainting or blackouts',
  'difficulty sleeping (too much, too little, early awakening)',
  'exaggerated symptoms or worse hangover from alcohol'],
 'cognitive/psych': ['difficulty with concentration or reading or "brain fog"',
  'confusion, difficulty thinking',
  'disorientation: getting lost; going to wrong places',
  'memory problems or forgetfulness',
  'mood swings, irritability, depression'],
 'musculo-skeletal': ['general/muscle weakness',
  'muscle pain or cramps',
  'joint pain or swelling',
  'swelling',
  'skin lesions (rash or lumpy lesions)'],
 'g.i': ['diarrhea',
  'bloating',
  'abdominal pain',
  'nausea',
  'vomiting',
  'indigestion or esophageal/"acid" reflux',
  'loss of appetite or unexplained weight loss'],
 'sexual/hormonal function': ['urinary incontinence or difficulty urinating',
  'unexplained m

In [20]:
mapping = dict(zip(data_dict["Sinai Description"], data_dict["Cardiff Name"]))
mapping

{'difficulty with concentration or reading or "brain fog"': 'poor_concentration',
 'confusion, difficulty thinking': 'nan',
 'memory problems or forgetfulness': 'nan',
 'fatigue or tiredness': 'fatigue',
 'cough': 'cough',
 'nan': 'nan',
 'shortness of breath': 'breathlessness_shortness_of_breath',
 'breathing faster than normal': 'nan',
 'headache': 'headache',
 'difficulty sleeping (too much, too little, early awakening)': 'sleep_disturbance',
 'general/muscle weakness': 'nan',
 'muscle pain or cramps': 'muscle_cramps',
 'joint pain or swelling': 'general_body_pain_e_g_arms_legs_or_joints',
 'loss of appetite or unexplained weight loss': 'loss_of_appetite',
 'nausea': 'nausea',
 'chest pain or discomfort': 'chest_pain',
 'heart palpitations, pulse skips, heart block': 'palpitations_heart_racing',
 'loss of smell': 'change_in_smell',
 'skin lesions (rash or lumpy lesions)': 'rash',
 'dizziness or lightheadedness': 'dizziness_or_vertigo',
 'tingling, numbness, burning, stabbing or "pin

In [21]:
for key in neurological_symptom_mapping.keys(): 
    values = neurological_symptom_mapping[key]
    for val in values: 
        if val not in list(data_dict["Sinai Description"]): 
            print(val)

In [22]:
for key in neurological_symptom_mapping.keys(): 
    values = []
    for val in neurological_symptom_mapping[key]: 
#         print(val)
        if mapping[val] != "nan": 
            values.append(mapping[val])
#         else: 
#             print(val)
    neurological_symptom_mapping[key] = values
neurological_symptom_mapping

{'neurological': ['headache',
  'change_in_vision',
  'fatigue',
  'dizziness_or_vertigo',
  'feeling_faint_or_light_headed',
  'sleep_disturbance'],
 'cognitive/psych': ['poor_concentration',
  'low_mood_lack_of_enjoyment_in_normal_activities'],
 'musculo-skeletal': ['muscle_cramps',
  'general_body_pain_e_g_arms_legs_or_joints',
  'rash'],
 'g.i': ['diarrhoea',
  'abdominal_pain',
  'nausea',
  'vomiting',
  'loss_of_appetite'],
 'sexual/hormonal function': ['pain_during_sex_having_intercourse'],
 'sensory changes': ['change_in_smell', 'numbness_odd_or_loss_of_sensation'],
 'temperature regulation': ['fevers_did_you_feel_hot',
  'high_temperature_when_you_need_it_measured',
  'increased_body_odour_sweating'],
 'pulmonary': ['cough', 'sore_throat', 'breathlessness_shortness_of_breath'],
 'cardiovascular': ['chest_pain', 'palpitations_heart_racing']}

In [23]:
# Actually averaging the missing values
print(data[list(data.isna().sum(axis = 1) != 0)].shape)
data[list(data.isna().sum(axis = 1) != 0)]

(23, 36)


,id,group,chest_pain,palpitations_heart_racing,poor_concentration,breathlessness_shortness_of_breath,cough,fevers_did_you_feel_hot,headache,abdominal_pain,change_in_smell,dizziness_or_vertigo,muscle_pain,poor_balance_unsteadiness,sleep_disturbance,increased_anxiety_worry,low_mood_lack_of_enjoyment_in_normal_activities,nausea,vomiting,muscle_cramps,constipation,diarrhoea,feeling_faint_or_light_headed,loss_of_appetite,back_pain,chills_do_you_feel_unusually_cold,numbness_odd_or_loss_of_sensation,increased_body_odour_sweating,general_body_pain_e_g_arms_legs_or_joints,pain_during_sex_having_intercourse,rash,runny_or_congested_nose,sore_throat,high_temperature_when_you_need_it_measured,change_in_vision,fatigue
35,CA036,Case,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
111,CA112,Case,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
122,CA123,Case,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1
125,CA126,Case,1,1,1,1,1,0,0,0,0,1,1,1,1,1,1,0,0,0,0,0,1,0,0,0,0,0,NaN,0,0,0,0,0,0,1
2,CO003,Control,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,CO006,Control,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,CO008,Control,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,CO009,Control,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12,CO013,Control,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14,CO015,Control,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
for index, row in data.iterrows(): 
    values = list(row)
    values = [str(val).lower() for val in values]
    if "nan" not in values: 
        continue
    if values.count('nan') < 33: 
        print(values)
        for c in range(len(data.columns)): 
            value = str(data.iloc[index, c]).lower()
            if value == 'nan': 
                col = list(data.columns)[c]
                print(col, value)
                for key in neurological_symptom_mapping.keys(): 
                    if col in neurological_symptom_mapping[key]: 
                        print(key, neurological_symptom_mapping[key])
                        print(list(row[neurological_symptom_mapping[key]]))
                        mean = np.nanmean(list(row[neurological_symptom_mapping[key]]))
                        print(mean)
                        print()
                        data.iloc[index, c] = mean

['ca123', 'case', '0', '0', '0', '1', '0', '0', '1', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', 'nan', '0', '0', '0', '0', '0', '0', '0', '0', '1', '0', '0', '0', '0', '0', '0', '1']
muscle_cramps nan
musculo-skeletal ['muscle_cramps', 'general_body_pain_e_g_arms_legs_or_joints', 'rash']
[nan, 1, 0]
0.5

['ca126', 'case', '1', '1', '1', '1', '1', '0', '0', '0', '0', '1', '1', '1', '1', '1', '1', '0', '0', '0', '0', '0', '1', '0', '0', '0', '0', '0', 'nan', '0', '0', '0', '0', '0', '0', '1']
general_body_pain_e_g_arms_legs_or_joints nan
musculo-skeletal ['muscle_cramps', 'general_body_pain_e_g_arms_legs_or_joints', 'rash']
[0, nan, 0]
0.0

['co043', 'control', '0', '1', '1', '1', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', 'nan', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '1']


In [25]:
data[list(data.isna().sum(axis = 1) != 0)]

,id,group,chest_pain,palpitations_heart_racing,poor_concentration,breathlessness_shortness_of_breath,cough,fevers_did_you_feel_hot,headache,abdominal_pain,change_in_smell,dizziness_or_vertigo,muscle_pain,poor_balance_unsteadiness,sleep_disturbance,increased_anxiety_worry,low_mood_lack_of_enjoyment_in_normal_activities,nausea,vomiting,muscle_cramps,constipation,diarrhoea,feeling_faint_or_light_headed,loss_of_appetite,back_pain,chills_do_you_feel_unusually_cold,numbness_odd_or_loss_of_sensation,increased_body_odour_sweating,general_body_pain_e_g_arms_legs_or_joints,pain_during_sex_having_intercourse,rash,runny_or_congested_nose,sore_throat,high_temperature_when_you_need_it_measured,change_in_vision,fatigue
35,CA036,Case,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
111,CA112,Case,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,CO003,Control,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,CO006,Control,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,CO008,Control,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,CO009,Control,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12,CO013,Control,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14,CO015,Control,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,CO018,Control,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN
19,CO02,Control,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [26]:
data = data.dropna(axis = 0, how = "any")
data

,id,group,chest_pain,palpitations_heart_racing,poor_concentration,breathlessness_shortness_of_breath,cough,fevers_did_you_feel_hot,headache,abdominal_pain,change_in_smell,dizziness_or_vertigo,muscle_pain,poor_balance_unsteadiness,sleep_disturbance,increased_anxiety_worry,low_mood_lack_of_enjoyment_in_normal_activities,nausea,vomiting,muscle_cramps,constipation,diarrhoea,feeling_faint_or_light_headed,loss_of_appetite,back_pain,chills_do_you_feel_unusually_cold,numbness_odd_or_loss_of_sensation,increased_body_odour_sweating,general_body_pain_e_g_arms_legs_or_joints,pain_during_sex_having_intercourse,rash,runny_or_congested_nose,sore_throat,high_temperature_when_you_need_it_measured,change_in_vision,fatigue
0,CA001,Case,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0
1,CA002,Case,1,1,0,1,1,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
2,CA003,Case,1,1,1,1,1,0,1,0,1,1,0,0,1,0,0,1,1,0,0,0,1,1,0,0,0,1,0,0,1,1,0,0,0,1
3,CA004,Case,0,0,0,0,0,0,1,1,0,1,0,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1
4,CA005,Case,1,1,1,1,1,0,0,0,0,1,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22,CO112,Control,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
23,CO113,Control,1,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
24,CO115,Control,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
25,CO117,Control,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [27]:
data["group"].value_counts()

Case       215
Control    102
Name: group, dtype: int64

In [28]:
# data.to_csv("data_cleaned/cardiff_data_all_n318.csv", index = False)